In [1]:
import sys
from pathlib import Path
import torch
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from MoE_Optimization.common.input_data_utils.mmlu_batch_generator import MMLUBatchGenerator





In [2]:
from transformers import AutoTokenizer, BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForCausalLM

from MoE_Optimization.common.clustering_utils import get_qwen


MODEL = "Qwen/Qwen1.5-MoE-A2.7B"


quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,

    bnb_4bit_compute_dtype=torch.float16,  # Compute in float16 for speed
    bnb_4bit_use_double_quant=True,  # Double quantization for extra memory savings
    # Normalized float 4-bit (optimal for LLMs)
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True,
                                            device_map='cuda:2',
                                            )
model = AutoModelForCausalLM.from_pretrained(MODEL, trust_remote_code=True,
                                            device_map='cuda:2',
                                            dtype=torch.float16,
                                            quantization_config=quantization_config,
                                            cache_dir="../../.cache"
                                            )

I0000 00:00:1778596936.462494   55088 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778596936.549416   55088 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778596938.005322   55088 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778596938.005854   55088 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [3]:
from MoE_Optimization.common.clustering_utils import SELECTED_TASKS
from MoE_Optimization.common.expert_utils.expert_clusterer import ExpertClusterer
from MoE_Optimization.common.input_data_utils.tokenization_utils import convert_text_inputs_to_tokens


gen = MMLUBatchGenerator(subjects=SELECTED_TASKS, split='validation', batch_size=16, include_metadata=False)
example_inputs = convert_text_inputs_to_tokens(gen, tokenizer, model.device)
clusterer = ExpertClusterer(model, example_inputs)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Initialized MMLU batch generator with 5 subjects
Split: validation, Batch size: 16

📚 Loading subject 1/5: anatomy


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✓ Created batch with 14 questions

📚 Loading subject 2/5: machine_learning


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✓ Created batch with 11 questions

📚 Loading subject 3/5: management


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✓ Created batch with 11 questions

📚 Loading subject 4/5: philosophy


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'cais/mmlu' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  ✓ Created batch with 16 questions

📚 Loading subject 5/5: world_religions
  ✓ Created batch with 16 questions


In [4]:
from MoE_Optimization.common.moe_utils import discover_moe_layers


layers = discover_moe_layers(model)
layers

[MoeHandle(layer_idx=0, moe=Qwen2MoeSparseMoeBlock(
   (gate): Linear4bit(in_features=2048, out_features=60, bias=False)
   (experts): ModuleList(
     (0-59): 60 x Qwen2MoeMLP(
       (gate_proj): Linear4bit(in_features=2048, out_features=1408, bias=False)
       (up_proj): Linear4bit(in_features=2048, out_features=1408, bias=False)
       (down_proj): Linear4bit(in_features=1408, out_features=2048, bias=False)
       (act_fn): SiLUActivation()
     )
   )
   (shared_expert): Qwen2MoeMLP(
     (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
     (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
     (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
     (act_fn): SiLUActivation()
   )
   (shared_expert_gate): Linear4bit(in_features=2048, out_features=1, bias=False)
 ), gate=Linear4bit(in_features=2048, out_features=60, bias=False), experts=ModuleList(
   (0-59): 60 x Qwen2MoeMLP(
     (gate_proj): Linear4bit(in_featu

In [ ]:
layer_idx2clusters = clusterer.create_clusters_by_front_PI_and_max_Calinski([layers[0], layers[1]])

In [6]:
handles2clusters

{0: [[45, 43, 51, 41, 48, 18, 31],
  [8, 25, 2, 1, 47, 53, 38],
  [29, 24, 42, 15],
  [30, 27],
  [40, 22],
  [39, 50],
  [20, 3],
  [36, 10],
  [11, 26],
  [44, 59, 0, 32, 55, 16, 35],
  [58, 7, 9, 14, 49, 5, 17, 13],
  [33, 28, 6, 19, 21, 46, 56],
  [34, 52, 12, 23],
  [4, 37],
  [57, 54]],
 1: [[25, 36, 50, 10, 13, 30, 40, 27, 26, 56, 34, 31, 37, 57, 15],
  [44, 43],
  [24, 14],
  [5, 12, 48],
  [39, 7, 45, 17, 58, 19, 54, 49],
  [3, 55, 46],
  [4, 9],
  [29, 6],
  [35, 0],
  [32, 21],
  [28, 41, 59, 8, 23, 11, 51],
  [33, 2],
  [22, 1],
  [18, 47],
  [52, 53],
  [38, 20],
  [42, 16]]}